# EM synapse mapping - multi-neuron use case

Creating a circuit from a set of morphologies with spines.

# Authentication

Follow the instructions below to authenticate and select a project to work in.

The morphology-with-spines instances that are to serve as the input of circuit generation must be accessible from the project.

In [1]:
import obi_auth

from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities

from entitysdk.models import CellMorphology, EMCellMesh, EMDenseReconstructionDataset, Circuit
from entitysdk import Client
from entitysdk._server_schemas import AssetLabel

token = obi_auth.get_token(environment="production", auth_mode="daf")
project_context = get_projects(token, env="production")

Rich is not supported, using fallback: No module named 'rich'


Device Authentication Required


1. Click on authentication URL

2. Complete authentication in browser

3. Return here when done


Authentication URL:

   https://cell-a.openbraininstitute.org/auth/realms/SBO/device?user_code=SWSK-GEBX
   ✓ Authentication completed successfully!


Dropdown(description='Select:', options=(('Single Cell Project', {'id': '50280a36-44b9-45e7-8b7d-db220a203ce5'…

# Find skeletonized morphologies as input

We search for morphologies originating from the `IARPA MICrONS mouse`, as they are derived by skeletonization

In [2]:
entity_client = Client(token_manager=token, project_context=project_context, environment="production")

results = entity_client.search_entity(
    entity_type=CellMorphology,
    query={"subject__name": "IARPA MICrONS mouse"},
).all()

# Filter by morphology name, avoid duplicates
morph_names = []
spiny_morphologies = []
for morph in results:
    if morph.name not in morph_names:
        morph_names.append(morph.name)
        spiny_morphologies.append(morph)

print(f"Found {len(spiny_morphologies)} spiny morphologies:")
for m in spiny_morphologies:
    print(f"  - {m.name} (id={m.id})")


Found 18 spiny morphologies:
  - MICrONS neuron 864691136330101226 (id=8c185dda-e7cb-40dd-a607-9a6b9e709744)
  - microns-neuron-864691136968109774 (id=219a2f26-6751-41c7-9517-af5a6c46152a)
  - microns-neuron-864691135773826043 (id=6141ecc4-0201-4cfd-9809-4712efa700f4)
  - Skeletonized-864691135491891559 (id=a57513b9-4ee4-4c81-99a8-ed487e43b545)
  - Skeletonized-864691135375351369 (id=c15ca1e6-6cf7-41c1-b52a-07328049684d)
  - Skeletonized-864691135592283403 (id=db6c83fd-4bfb-4fc9-b9b5-eb43edc96bdd)
  - Skeletonized-864691136967478222 (id=8d1841ae-c46a-4f0a-bf29-513045cd6f38)
  - Skeletonized-864691136437883550 (id=251f3101-aa0d-4cac-a243-6a6e5d7639a8)
  - Skeletonized-864691135473464114 (id=26e1e6c0-c624-40c4-84a2-c72ab22e712d)
  - Skeletonized-864691135105464653 (id=c8a236b4-930a-4cfd-a14f-17ccdc9cfe3f)
  - Skeletonized-864691135782490704 (id=863aaa19-df28-45fc-b9f8-ed4466d8d707)
  - Skeletonized-864691136967869902 (id=add5d3fb-9f8d-446c-a039-dd8fddd73869)
  - Skeletonized-864691135274

## Get `pt_root_id`. 
Next, we have to link the skeletons to their originating cell mesh. In the absence of an activity linking the two, we match them via the identifier, `pt_root_id`.

We print the description of the neurons below.

In [3]:
from entitysdk.models import SkeletonizationExecution, EMCellMesh
from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID
from obi_one.scientific.tasks.em_synapse_mapping.resolve_neuron import resolve_provenance

# {pt_root_id: {"morph_from_id": CellMorphologyFromID, "name": str}}
neurons = {}

for morph_entity in spiny_morphologies:
    morph_from_id = CellMorphologyFromID(id_str=str(morph_entity.id))

    if not morph_from_id.has_source_mesh(db_client=entity_client):
        print(f"  WARNING: Could not resolve provenance for {morph_entity.name}, skipping.")
        continue

    pt_root_id, source_mesh, _ = resolve_provenance(entity_client, morph_from_id)
    if source_mesh.release_version > 1412:

        print(f"{morph_entity.description}")
        print(f"  {morph_entity.name} -> pt_root_id={pt_root_id}")
        neurons[pt_root_id] = {"morph_from_id": morph_from_id, "name": morph_entity.name}

print(f"\nTotal neuron entries: {len(neurons)}")

[2026-06-10 11:53:25,427] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-morphology/8c185dda-e7cb-40dd-a607-9a6b9e709744 "HTTP/1.1 200 OK"
[2026-06-10 11:53:25,709] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=8c185dda-e7cb-40dd-a607-9a6b9e709744&page=1 "HTTP/1.1 200 OK"
[2026-06-10 11:53:25,996] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=8c185dda-e7cb-40dd-a607-9a6b9e709744&page=1 "HTTP/1.1 200 OK"
[2026-06-10 11:53:26,284] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=8c185dda-e7cb-40dd-a607-9a6b9e709744&page=1 "HTTP/1.1 200 OK"
[2026-06-10 11:53:26,502] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-cell-mesh/7d3441c4-0314-411a-aaf2-59a82c1d087e "HTTP/1.1 200 OK"
[2026-06-10 11:53:26,719] INFO: HTTP Req

## Inspect connectivity between neurons

Query CAVE for synapses among the resolved neurons and show, for each physical neuron, which other neurons it connects to (pre- and post-synaptically).

First, create a CAVE token

In [4]:
import caveclient
temporary_client = caveclient.CAVEclient()
temporary_client.auth.get_new_token()

New Tokens need to be acquired by hand. Please follow the following steps:
                1) Go to: https://global.daf-apis.com/auth/api/v1/create_token to create a new token.
                2) Log in with your Google credentials and copy the token shown afterward.
                3a) Save it to your computer with: client.auth.save_token(token="PASTE_YOUR_TOKEN_HERE")
                or
                3b) Set it for the current session only with client.auth.token = "PASTE_YOUR_TOKEN_HERE"
                Note: If you need to save or load multiple tokens, please read the documentation for details.
                Warning! Creating a new token by finishing step 2 will invalidate the previous token!


Paste your token below:

In [5]:
import os
import caveclient
os.environ["CAVE_TOKEN"] = "dac7bc9fc59ac3555044d4c9d878927e"
os.environ["CAVECLIENT_MICRONS_API_KEY"] = "dac7bc9fc59ac3555044d4c9d878927e"

Inspect connectivity between the neurons in the set.

In [6]:
# Ignore caveclient/materializationengine.py:1745 -- RuntimeWarning
import warnings
warnings.filterwarnings("ignore", message="Engine has switched to", category=RuntimeWarning)

CAVE_DATASTACK = "minnie65_public"

cave = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CAVE_TOKEN"])
cave.version = 1507
pt_root_ids = list(neurons.keys())

assert cave.materialize.version == 1507

syns = cave.materialize.synapse_query(post_ids=pt_root_ids, pre_ids=pt_root_ids)
raw_mat = syns.groupby(["pre_pt_root_id", "post_pt_root_id"])["valid"].count().unstack("post_pt_root_id", fill_value=0)
mat = raw_mat.reindex(columns=pt_root_ids, index=pt_root_ids).fillna(0).to_numpy()
mat

array([[0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 1, 1, 0.0, 0.0, 4, 0.0, 1, 0, 0.0, 0.0, 1],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 1],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 1, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [1, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0, 0.0, 0, 0, 0.0, 0.0, 0],
       [0, 0.0, 0.0, 0.0,

Now we pick the five neurons with the highest degrees.

In [7]:
import numpy

#degrees
idxx = numpy.argsort(mat.sum(axis=0) + mat.sum(axis=1))[-5:]
circuit_pt_root_ids = numpy.array(pt_root_ids)[idxx]

mat[:, idxx][idxx]

array([[0.0, 1, 0, 0, 0.0],
       [0.0, 0, 0, 0, 0.0],
       [0.0, 0, 0, 0, 0.0],
       [0.0, 0, 0, 0, 0.0],
       [0.0, 1, 1, 4, 0.0]], dtype=object)

In [8]:
circuit_neurons = [neurons[pt_id]["morph_from_id"] for pt_id in circuit_pt_root_ids]
print(circuit_pt_root_ids)

[864691136006519498 864691136437883550 864691136603649745
 864691136967869902 864691135773826043]


# Set up the multi-neuron synapse mapping task

In [11]:
from obi_one.scientific.tasks.em_synapse_mapping.config import EMSynapseMappingSingleConfig, AdvancedEMSynapseMappingOptions
from obi_one.scientific.tasks.em_synapse_mapping.task import EMSynapseMappingTask
from obi_one.scientific.from_id.named_tuple_from_id import EMSynapseMappingInputNamedTuple
from obi_one.core.info import Info

# We take the last 3 digits of pt_root_id to easily identify the neurons
circuit_neurons_short_names = [str(pt_id)[-3:] for pt_id in circuit_pt_root_ids]

OUTPUT_DIR = "multi-neuron-circuit-" + "-".join([id for id in circuit_neurons_short_names])

task_config = EMSynapseMappingSingleConfig(
    info=Info(
        campaign_name="EM Synapse Mapping Multi-Neuron",
        campaign_description="Map EM synapses onto multiple spiny morphologies"
    ),
    initialize=EMSynapseMappingSingleConfig.Initialize(
        neurons=EMSynapseMappingInputNamedTuple(name="Skeletons", elements=tuple(circuit_neurons)),
    ),
    coordinate_output_root=OUTPUT_DIR,
    advanced_options=AdvancedEMSynapseMappingOptions()
)

task = EMSynapseMappingTask(config=task_config)
print(f"Running synapse mapping task with {len(circuit_neurons)} neurons...")
print(f"Output: {OUTPUT_DIR}/")
task.execute(db_client=entity_client)

Running synapse mapping task with 5 neurons...
Output: multi-neuron-circuit-498-550-745-902-043/
[2026-06-10 11:59:09,728] INFO: Preparing output at multi-neuron-circuit-498-550-745-902-043...
[2026-06-10 11:59:09,730] INFO: Resolving neurons...
[2026-06-10 11:59:09,730] INFO: Placing morphologies...
[2026-06-10 11:59:10,121] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-morphology/15740f5b-8906-4f84-b426-6bc5a09e6ae2/assets/fef910bc-6962-46dc-87e6-c9d7e21c9fdf "HTTP/1.1 200 OK"
[2026-06-10 11:59:10,231] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-morphology/15740f5b-8906-4f84-b426-6bc5a09e6ae2/assets/fef910bc-6962-46dc-87e6-c9d7e21c9fdf/download "HTTP/1.1 307 Temporary Redirect"
[2026-06-10 11:59:10,701] INFO: HTTP Request: GET https://entitycore-data-production.s3.amazonaws.com/private/81f810d7-62e6-4431-a16a-f812ea3624be/d94547f2-120b-4c57-8d7e-0924b5d5765e/assets/cell_morphology/15740f5b-8906-4f84-b426-6bc5a0

/Users/runner/work/MorphIO/MorphIO/3rdparty/HighFive/include/highfive/bits/H5ReadWrite_misc.hpp: 154 [WARN] /spines/skeletons/microns-mesh-864691136006519498/points": hdf5 dataset has higher floating point precision than data on read: Float64 -> Float32

HDF5 GROUP:0:warning


[2026-06-10 12:02:16,647] INFO: Resolving skeleton provenance...
[2026-06-10 12:02:16,932] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=15740f5b-8906-4f84-b426-6bc5a09e6ae2&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:02:17,220] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=15740f5b-8906-4f84-b426-6bc5a09e6ae2&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:02:17,441] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-cell-mesh/ab751398-51c1-4f7d-a4f0-6de3aa2b239e "HTTP/1.1 200 OK"
[2026-06-10 12:02:17,563] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-dense-reconstruction-dataset/6ee16ab4-3068-46e6-86c9-4e1f6dde5927 "HTTP/1.1 200 OK"
[2026-06-10 12:02:17,566] INFO: Placing morphologies...
[2026-06-10 12:02:17,672] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-

/Users/runner/work/MorphIO/MorphIO/3rdparty/HighFive/include/highfive/bits/H5ReadWrite_misc.hpp: 154 [WARN] /spines/skeletons/microns-mesh-864691136437883550/points": hdf5 dataset has higher floating point precision than data on read: Float64 -> Float32

HDF5 GROUP:0:warning


[2026-06-10 12:06:37,205] INFO: Resolving skeleton provenance...
[2026-06-10 12:06:37,770] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=251f3101-aa0d-4cac-a243-6a6e5d7639a8&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:06:38,056] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=251f3101-aa0d-4cac-a243-6a6e5d7639a8&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:06:38,270] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-cell-mesh/151b14e0-85fc-4d56-807c-78c35f684a03 "HTTP/1.1 200 OK"
[2026-06-10 12:06:38,387] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-dense-reconstruction-dataset/6ee16ab4-3068-46e6-86c9-4e1f6dde5927 "HTTP/1.1 200 OK"
[2026-06-10 12:06:38,389] INFO: Placing morphologies...
[2026-06-10 12:06:38,494] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-

...

[2026-06-10 12:10:02,950] INFO: Resolving skeleton provenance...
[2026-06-10 12:10:03,577] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=903c0864-716b-4d6c-adfe-b58b7057aa05&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:10:03,907] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=903c0864-716b-4d6c-adfe-b58b7057aa05&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:10:04,138] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-cell-mesh/de860a5c-0381-47df-b949-8cc707978cc9 "HTTP/1.1 200 OK"
[2026-06-10 12:10:04,260] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-dense-reconstruction-dataset/6ee16ab4-3068-46e6-86c9-4e1f6dde5927 "HTTP/1.1 200 OK"
[2026-06-10 12:10:04,263] INFO: Placing morphologies...
[2026-06-10 12:10:04,371] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-

/Users/runner/work/MorphIO/MorphIO/3rdparty/HighFive/include/highfive/bits/H5ReadWrite_misc.hpp: 154 [WARN] /spines/skeletons/microns-mesh-864691136967869902/points": hdf5 dataset has higher floating point precision than data on read: Float64 -> Float32

HDF5 GROUP:0:warning


[2026-06-10 12:12:48,696] INFO: Resolving skeleton provenance...
[2026-06-10 12:12:48,978] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=add5d3fb-9f8d-446c-a039-dd8fddd73869&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:12:49,365] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=add5d3fb-9f8d-446c-a039-dd8fddd73869&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:12:49,585] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-cell-mesh/15691f2e-4105-4f73-9066-50ec1e469768 "HTTP/1.1 200 OK"
[2026-06-10 12:12:49,702] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-dense-reconstruction-dataset/6ee16ab4-3068-46e6-86c9-4e1f6dde5927 "HTTP/1.1 200 OK"
[2026-06-10 12:12:49,705] INFO: Placing morphologies...
[2026-06-10 12:12:49,813] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/cell-

/Users/runner/work/MorphIO/MorphIO/3rdparty/HighFive/include/highfive/bits/H5ReadWrite_misc.hpp: 154 [WARN] /morphology/microns-mesh-864691135773826043/points": hdf5 dataset has higher floating point precision than data on read: Float64 -> Float32
/Users/runner/work/MorphIO/MorphIO/3rdparty/HighFive/include/highfive/bits/H5ReadWrite_misc.hpp: 154 [WARN] /spines/skeletons/microns-mesh-864691135773826043/points": hdf5 dataset has higher floating point precision than data on read: Float64 -> Float32

HDF5 GROUP:0:warning


[2026-06-10 12:12:54,813] INFO: Resolving skeleton provenance...
[2026-06-10 12:12:55,110] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=6141ecc4-0201-4cfd-9809-4712efa700f4&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:12:55,400] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/skeletonization-execution?generated__id=6141ecc4-0201-4cfd-9809-4712efa700f4&page=1 "HTTP/1.1 200 OK"
[2026-06-10 12:12:55,621] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-cell-mesh/07aff2d6-e993-4cab-8bc0-4b0b5763859b "HTTP/1.1 200 OK"
[2026-06-10 12:12:55,740] INFO: HTTP Request: GET https://cell-a.openbraininstitute.org/api/entitycore/em-dense-reconstruction-dataset/6ee16ab4-3068-46e6-86c9-4e1f6dde5927 "HTTP/1.1 200 OK"
[2026-06-10 12:12:55,743] INFO: Merging spiny morphologies into combined file...
[2026-06-10 12:12:55,797] INFO: Reading data from source EM reconstructions...
[202

## Inspect the output (local files)

In [12]:
import json
import h5py
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
config_path = output_dir / "circuit_config.json"

if config_path.exists():
    with open(config_path) as f:
        circuit_cfg = json.load(f)
    print(json.dumps(circuit_cfg, indent=2))

    edges_file = output_dir / "synaptome-edges.h5"
    if edges_file.exists():
        with h5py.File(edges_file, "r") as h5:
            print("\nEdge populations:")
            for pop_name in h5["edges"]:
                grp = h5[f"edges/{pop_name}"]
                n = len(grp["source_node_id"])
                src_pop = grp["source_node_id"].attrs.get("node_population", "?")
                tgt_pop = grp["target_node_id"].attrs.get("node_population", "?")
                print(f"  {pop_name}: {n} edges ({src_pop} -> {tgt_pop})")
else:
    print("No circuit output found. Run the task cell above first.")

{
  "components": {
    "biophysical_neuron_models_dir": "",
    "mechanisms_dir": "",
    "morphologies_dir": "",
    "point_neuron_models_dir": "",
    "synaptic_models_dir": "",
    "templates_dir": ""
  },
  "networks": {
    "edges": [
      {
        "edges_file": "$BASE_DIR/synaptome-edges.h5",
        "populations": {
          "physical_connections": {
            "type": "chemical"
          },
          "virtual_afferents": {
            "type": "chemical"
          }
        }
      }
    ],
    "nodes": [
      {
        "nodes_file": "$BASE_DIR/synaptome-nodes.h5",
        "populations": {
          "biophysical_neurons": {
            "biophysical_neuron_models_dir": "$BASE_DIR/hoc",
            "morphologies_dir": "$BASE_DIR/morphologies",
            "type": "biophysical",
            "alternate_morphologies": {
              "h5v1": "$BASE_DIR/morphologies/merged_spiny_morphologies.h5"
            }
          }
        }
      },
      {
        "nodes_file": "$BASE_D

## Test with SNAP

In [13]:
import bluepysnap as snap

circ = snap.Circuit("multi-neuron-circuit-498-550-745-902-043/circuit_config.json")
bio_pop = circ.nodes["biophysical_neurons"]
bio_pop.get()

,cell_type,morphology,pt_root_id,status_axon,status_dendrite,synapse_class,volume,x,y,z
node_ids,,,,,,,,,,
0,L4b,morphology/15740f5b-8906-4f84-b426-6bc5a09e6ae2,864691136006519498,True,True,excitatory_neuron,224.408859,712.384,635.776,887.88
1,L4c,morphology/251f3101-aa0d-4cac-a243-6a6e5d7639a8,864691136437883550,True,True,excitatory_neuron,278.539154,718.720,736.448,889.72
2,L4c,morphology/903c0864-716b-4d6c-adfe-b58b7057aa05,864691136603649745,True,True,excitatory_neuron,258.118134,718.656,719.872,893.80
3,L6tall-a,morphology/add5d3fb-9f8d-446c-a039-dd8fddd73869,864691136967869902,True,True,excitatory_neuron,207.492554,714.816,902.464,886.24
4,DTC,morphology/6141ecc4-0201-4cfd-9809-4712efa700f4,864691135773826043,True,True,inhibitory_neuron,243.582077,737.728,465.664,845.44


Morphologies resolve (swc)

In [14]:
bio_pop.morph.get(0, extension="swc")

Morphologies resolve (h5)

In [15]:
bio_pop.morph.get(0, extension="h5")

Spiny morphologies load

In [16]:
from morph_spines import load_morphology_with_spines

load_morphology_with_spines(bio_pop.config["alternate_morphologies"]["h5v1"],
                            bio_pop.get(0, properties="morphology").split("/")[1])

/Users/runner/work/MorphIO/MorphIO/3rdparty/HighFive/include/highfive/bits/H5ReadWrite_misc.hpp: 154 [WARN] /spines/skeletons/microns-mesh-864691136006519498/points": hdf5 dataset has higher floating point precision than data on read: Float64 -> Float32

HDF5 GROUP:0:warning


MorphologyWithSpines(morphology=Morphology <soma: SomaSimpleContour(array([[7.13752e+02, 6.29937e+02, 8.88011e+02, 2.83340e-01],
       [7.13578e+02, 6.30539e+02, 8.88041e+02, 3.19137e-01],
       [7.13404e+02, 6.31140e+02, 8.88070e+02, 9.19609e-01],
       [7.13230e+02, 6.31742e+02, 8.88100e+02, 1.52839e+00],
       [7.13056e+02, 6.32344e+02, 8.88129e+02, 2.09831e+00],
       [7.12882e+02, 6.32945e+02, 8.88159e+02, 2.61862e+00],
       [7.12708e+02, 6.33547e+02, 8.88188e+02, 2.99914e+00],
       [7.12534e+02, 6.34149e+02, 8.88218e+02, 3.18773e+00],
       [7.12360e+02, 6.34751e+02, 8.88248e+02, 3.37043e+00],
       [7.12186e+02, 6.35352e+02, 8.88277e+02, 3.61296e+00],
       [7.12012e+02, 6.35954e+02, 8.88307e+02, 3.85225e+00],
       [7.11838e+02, 6.36556e+02, 8.88336e+02, 3.72451e+00],
       [7.11664e+02, 6.37158e+02, 8.88366e+02, 3.48372e+00],
       [7.11490e+02, 6.37759e+02, 8.88395e+02, 3.09590e+00],
       [7.11315e+02, 6.38361e+02, 8.88425e+02, 2.63015e+00],
       [7.11141e+